## Inference on QA Dataset (dev.jsonl -> QA)

* google/bigbird-roberta-base w/o DA

In [4]:
import json
from string import Template
import numpy as np
import torch
from tqdm import tqdm
from collections import Counter
import torch.nn.functional as F
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, classification_report
from transformers import AutoTokenizer, AutoModelForMaskedLM
import json
import os

/home/mario/miniforge3/envs/xtemp-nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [17]:
model_id = 'google/bigbird-roberta-base'

model = AutoModelForMaskedLM.from_pretrained(model_id, trust_remote_code=True, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

BigBirdForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
/home/mario/miniforge3/envs/xtemp-nlp/lib/python3.10/site-packages/torch/cuda/__init__.py:1007: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml

In [11]:
def score_choice(choice, model, tokenizer):
    model.eval()
    device = model.device

    with torch.no_grad():
        q, a = choice.split(' <sep> ')
        q, a = q.strip(), a.strip()

        enc = tokenizer(q, a, padding='max_length', truncation=True, max_length=128, return_tensors='pt')

        input_ids, attn_mask = enc.input_ids.to(device), enc.attention_mask.to(device)

        is_answer, answer_pos = False, []
        for idx, input_id in enumerate(input_ids[0]):
            # first [SEP]
            if not is_answer and input_id.item() == tokenizer.sep_token_id:
                is_answer = True
                continue
            # final [SEP]
            elif is_answer and input_id.item() == tokenizer.sep_token_id:
                break
            # answer is in-between [SEP] tokens
            if is_answer:
                answer_pos.append(idx)

        batch_input_ids, batch_attn_mask, target_token_ids = [], [], []
        for idx in answer_pos:
            token_id_original = input_ids[0, idx].item()

            masked = input_ids.clone()
            masked[0, idx] = tokenizer.mask_token_id

            batch_input_ids.append(masked[0])
            batch_attn_mask.append(attn_mask[0])
            target_token_ids.append(token_id_original)

        batch_input_ids = torch.stack(batch_input_ids).to(device)
        batch_attn_mask = torch.stack(batch_attn_mask).to(device)
        target_token_ids = torch.tensor(target_token_ids).to(device)

        outputs = model(input_ids=batch_input_ids, attention_mask=batch_attn_mask)

        logits = outputs.logits
        log_probs = F.log_softmax(logits, dim=-1)

        token_logprobs = []

        for i, idx in enumerate(answer_pos):
            token_logprob = log_probs[i, idx, target_token_ids[i]].item()
            token_logprobs.append(token_logprob)

        logprob = sum(token_logprobs) / len(token_logprobs)

        return logprob, np.exp(logprob)

In [ ]:
test_path = '../data/baseline/test.json'

with open(test_path, 'r') as f:
    test_data = json.load(f)

In [ ]:
prompt_template = Template('$question <sep> $answer')
Y, Y_hat = [], []

## this can also be random, but then is harder to assess whether its effective
print(f"\n\n*** Evaluate QA ***\nNum Questions : {len(test_data)}\n\n")

for test_example in tqdm(test_data, desc='Inference on full QA dataset'):
    choices = [
        prompt_template.substitute(question=test_example['question'], answer=test_example[f'op{clet}']) for
        clet
        in ['a', 'b', 'c', 'd']]

    cop = test_example['cop']
    # print(f'gold answer: {cop}')
    max_prob, ans = float('-inf'), None
    for cop_idx in range(len(choices)):
        logs, _ = score_choice(choices[cop_idx], model, tokenizer)
        # print(f'cop{cop_idx + 1}: {logs}')
        if max_prob < logs:
            max_prob = logs
            ans = cop_idx + 1
    Y.append(cop)
    Y_hat.append(ans)

labels = [1, 2, 3, 4]

p_micro = precision_score(Y, Y_hat, average='micro', zero_division=0)
r_micro = recall_score(Y, Y_hat, average='micro', zero_division=0)
f1_micro = f1_score(Y, Y_hat, average='micro', zero_division=0)

p_macro = precision_score(Y, Y_hat, average='macro', zero_division=0, labels=labels)
r_macro = recall_score(Y, Y_hat, average='macro', zero_division=0, labels=labels)
f1_macro = f1_score(Y, Y_hat, average='macro', zero_division=0, labels=labels)

acc = accuracy_score(Y, Y_hat)

report = classification_report(Y, Y_hat, zero_division=0, labels=labels,
                               target_names=["opa", "opb", "opc", "opd"])

print(f'Gold Ans: {Counter(Y)}')
print(f'Pred Ans: {Counter(Y_hat)}')

print(report)

d = {'P-micro': p_micro, 'R-micro': r_micro, 'F1-micro': f1_micro, 'P-macro': p_macro, 'R-macro': r_macro,
     'F1-macro': f1_macro, 'Acc': acc}

print(d)

